In [5]:
import pandas
import math
import json

with open('setup.json') as f:
    data = json.load(f)
path = data["describe_path"]
print(f'path : {path}')

path : /nfs/homes/lfarina/Desktop/ft_dslr/datasets/dataset_train.csv


In [6]:
def csv_to_dataframe(csv_file: str) -> pandas.DataFrame:
    """Converts a CSV file into a Pandas dataframe"""
    dtf = pandas.read_csv(csv_file)
    return dtf
csv_to_dataframe(path)

,Index,Hogwarts House,First Name,Last Name,Birthday,Best Hand,Arithmancy,Astronomy,Herbology,Defense Against the Dark Arts,Divination,Muggle Studies,Ancient Runes,History of Magic,Transfiguration,Potions,Care of Magical Creatures,Charms,Flying
0,0,Ravenclaw,Tamara,Hsu,2000-03-30,Left,58384.0,-487.886086,5.727180,4.878861,4.722,272.035831,532.484226,5.231058,1039.788281,3.790369,0.715939,-232.79405,-26.89
1,1,Slytherin,Erich,Paredes,1999-10-14,Right,67239.0,-552.060507,-5.987446,5.520605,-5.612,-487.340557,367.760303,4.107170,1058.944592,7.248742,0.091674,-252.18425,-113.45
2,2,Ravenclaw,Stephany,Braun,1999-11-03,Left,23702.0,-366.076117,7.725017,3.660761,6.140,664.893521,602.585284,3.555579,1088.088348,8.728531,-0.515327,-227.34265,30.42
3,3,Gryffindor,Vesta,Mcmichael,2000-08-19,Left,32667.0,697.742809,-6.497214,-6.977428,4.026,-537.001128,523.982133,-4.809637,920.391449,0.821911,-0.014040,-256.84675,200.64
4,4,Gryffindor,Gaston,Gibbs,1998-09-27,Left,60158.0,436.775204,-7.820623,NaN,2.236,-444.262537,599.324514,-3.444377,937.434724,4.311066,-0.264070,-256.38730,157.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,1595,Gryffindor,Jung,Blank,2001-09-14,Right,49009.0,354.280086,-4.541837,-3.542801,5.702,-497.235066,618.220213,-5.231721,964.219853,3.389086,-0.649983,-250.39401,185.83
1596,1596,Slytherin,Shelli,Lock,1998-03-12,Left,63296.0,367.531174,6.061064,-3.675312,1.757,-643.271092,445.827565,2.238112,1056.147366,5.825263,-0.333962,-246.42719,44.80
1597,1597,Gryffindor,Benjamin,Christensen,1999-10-24,Right,63905.0,544.018925,-3.203269,-5.440189,6.065,-385.150457,635.211486,-5.984257,953.866685,1.709808,0.071569,-251.63679,198.47
1598,1598,Hufflepuff,Charlotte,Dillon,2001-09-21,Left,82713.0,453.676219,3.442831,-4.536762,6.738,-831.741123,383.444937,3.813111,1087.949205,3.904100,-0.531875,-246.19072,-76.81


In [7]:
def describe_csv(dtf: pandas.DataFrame, verbose: bool = True):
    """Interprets a Pandas dataframe to analyse basic information like the mean, the standard deviation or the min and max value"""
    describe_dtf = pandas.DataFrame()
    dtf = dtf.drop(columns='Index', inplace=False)
    nb_features = dtf.select_dtypes(include=['number']).columns.tolist()
    describe_dtf['info'] = ['Count', 'Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max']

    for feature in nb_features:
        try:
            describe_dtf[feature] = describe_column_nb(dtf, feature)
        except Exception:
            pass
    if verbose:
        print(describe_dtf.to_string(index=False))
    return describe_dtf

In [8]:
def describe_column_nb(dtf: pandas.DataFrame, column: str) -> []:
    """Get the description of a column"""
    dtf_cleaned = pandas.DataFrame(dtf[column].copy())
    dtf_cleaned.dropna(inplace=True)
    dtf_cleaned.sort_values(by=column, ascending=True, inplace=True)
    dtf_cleaned.reset_index(drop=True, inplace=True)

    count, mean, sum_v, min_v, max_v = extract_basics_info(dtf_cleaned, column)
    std = compute_std(dtf_cleaned, column, mean, count)
    twenty_five = compute_quantile(dtf_cleaned, column, count, 0.25)
    fifty = compute_quantile(dtf_cleaned, column, count, 0.5)
    seventy_five = compute_quantile(dtf_cleaned, column, count, 0.75)

    return count, mean, std, min_v, twenty_five, fifty, seventy_five, max_v

In [9]:
def extract_basics_info(dtf: pandas.DataFrame, column: str) -> []:
    """Extract the maximum information of a column in a single loop.
    :return : count of items, the mean, the total sum, the min and max value"""
    count, mean, sum_v, min_v, max_v = 0, 0, 0, 0, 0
    for index, value in dtf[column].items():
        count = count + 1
        if count == 1:
            min_v = value
            max_v = value
        if value < min_v:
            min_v = value
        if value > max_v:
            max_v = value
        sum_v += value

    mean = sum_v / count
    return count, mean, sum_v, min_v, max_v

In [10]:
def compute_std(dtf: pandas.DataFrame, column: str, mean: float, count: int) -> []:
    """Compute the standard deviation of a column"""
    sum_tmp = 0
    for index, value in dtf[column].items():
        sum_tmp = sum_tmp + (value - mean) * (value - mean)
    std = math.sqrt(sum_tmp / count)
    return std

In [11]:
def compute_quantile(dtf: pandas.DataFrame, column: str, count: int, quantile: float) -> []:
    """Compute the quantile of a column"""
    quantile_idx = quantile * count                 # find the quantile index (expressed in float)
    if quantile_idx.is_integer():                   # access directly to the value if the quantile_idx can be an int
        quantile_value = dtf.loc[quantile_idx, column]
    elif quantile_idx - int(quantile_idx) < 0.5:    # find the nearst value if the quantile_idx cant be an int
        quantile_value = dtf.loc[int(quantile_idx), column]
    else:
        quantile_value = dtf.loc[math.ceil(quantile_idx), column]
    return quantile_value

In [12]:
dtf = csv_to_dataframe(path)
describe_csv(dtf, verbose=False)

,info,Arithmancy,Astronomy,Herbology,Defense Against the Dark Arts,Divination,Muggle Studies,Ancient Runes,History of Magic,Transfiguration,Potions,Care of Magical Creatures,Charms,Flying
0,Count,1566.000000,1568.000000,1567.000000,1569.000000,1561.00000,1565.000000,1565.000000,1557.000000,1566.000000,1570.000000,1560.000000,1600.000000,1600.000000
1,Mean,49634.570243,39.797131,1.141020,-0.387863,3.15391,-224.589915,495.747970,2.963095,1030.096946,5.950373,-0.053427,-243.374409,21.958013
2,Std,16674.479577,520.132330,5.218016,5.211132,4.15397,486.189433,106.251202,4.424353,44.111025,3.146852,0.971146,8.780895,97.601087
3,Min,-24370.000000,-966.740546,-10.295663,-10.162119,-8.72700,-1086.496835,283.869609,-8.858993,906.627320,-4.697484,-3.313676,-261.048920,-181.470000
4,25%,38516.000000,-489.493777,-4.304245,-5.259095,3.09900,-577.580096,397.511047,2.218653,1026.324832,3.652442,-0.670996,-250.647270,-41.840000
5,50%,49018.000000,261.644731,3.486307,-2.572919,4.62500,-418.660994,464.327598,4.379618,1045.533335,5.877582,-0.043296,-244.867510,-2.510000
6,75%,60918.000000,525.909540,5.421046,4.906820,5.66800,258.777525,597.703964,5.836217,1058.485415,8.254311,0.594446,-232.536750,50.890000
7,Max,104956.000000,1016.211940,11.612895,9.667405,10.03200,1092.388611,745.396220,11.889713,1098.958201,13.536762,3.056546,-225.428140,279.070000
